# 16. Reproducible scientific evaluators and numerical contracts

![Reproducible evaluator](../images/16_reproducible_scientific_evaluators.svg)

This notebook builds three of the boundaries from the lecture and then tries to break them: a canonical digest, a registry validator that checks the study rather than the row, and a retrieval score that GFC and its control can share.

Read the [lecture](../lectures/16_reproducible_scientific_evaluators.md) alongside it, and return to the [tutorial index](../README.md).

**Learning goals:** create a canonical digest, validate the 28-model allocation registry, reject provenance mismatches, and verify that GFC and independent completion use a comparable gallery margin.

In [ ]:
import hashlib
import json
import numpy as np

SEED = 31
rng = np.random.default_rng(SEED)
assert rng.integers(0, 2) in {0, 1}


## Canonical bytes

A digest answers one narrow question: are these the exact bytes I saw before? To make the answer stable, serialize with sorted keys and fixed separators so that two dictionaries with the same content produce the same hash regardless of insertion order. The assertion below is that property.

Identity is not sense. A digest cannot tell you that the registry it fingerprints describes a sound experiment, so the next section validates meaning separately.

In [ ]:
def digest(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(payload).hexdigest()

assert digest({'b': 2, 'a': 1}) == digest({'a': 1, 'b': 2})


## The 28-row registry

Eight blocks each contain breadth, balanced, and phase depth. Four prespecified blocks add nearby jitter. That is `8 × 3 + 4 = 28` rows, and the count is asserted below.

Look at what a row stores. Alongside the filenames it records the scientific meaning of the run: how many sequences, how many origins per sequence, the resulting nominal catalog size, the origin policy, the planned exposure, the paired seeds, and a digest for every frozen stream. A row that stores only paths cannot be validated later.

In [ ]:
ALLOCATIONS = {
    'breadth': (250_000, 1, 'base_phase'),
    'balanced': (125_000, 2, 'phase_separated'),
    'phase_depth': (62_500, 4, 'phase_separated'),
    'nearby_jitter': (62_500, 4, 'nearby_jitter'),
}

def make_row(block, allocation):
    unique_sequences, origins_per_sequence, origin_policy = ALLOCATIONS[allocation]
    paired_pool = 'phase_depth' if allocation == 'nearby_jitter' else allocation
    return {
        'block': block, 'allocation': allocation,
        'unique_sequences': unique_sequences, 'origins_per_sequence': origins_per_sequence,
        'nominal_catalog_size': unique_sequences * origins_per_sequence,
        'origin_policy': origin_policy, 'planned_exposure': 4_096_000,
        'optimization_seed': 100 + block, 'replicate_seed': 200 + block,
        'train_manifest_digest': digest({'block': block, 'allocation': allocation}),
        'sequence_pool_digest': digest({'block': block, 'pool': paired_pool}),
        'source_group_digest': digest({'block': block, 'pool': paired_pool, 'groups': 'v1'}),
        'phase_catalog_digest': 'phase-v1',
        'sequence_stream_version': 'sequence-v2', 'phase_stream_version': 'phase-v1',
        'spatial_stream_version': 'spatial-v1', 'mask_stream_version': 'mask-v1',
    }

registry = [make_row(block, allocation) for block in range(8) for allocation in ('breadth', 'balanced', 'phase_depth')]
registry += [make_row(block, 'nearby_jitter') for block in range(4)]
assert len(registry) == 28


## Cross-row checks

Every row here can be individually valid while the registry as a whole is wrong, so the validator works on relationships. It compares the observed set of block and allocation pairs against the complete expected set, which catches a duplicated cell hiding a missing one. It then checks that every row reaches the same nominal catalog, and that each jitter row shares its selected sequence pool, source groups, exposure, seeds, phase catalog, and nuisance streams with the phase-depth row it is paired against.

The second half of the cell is the important half: it mutates one valid registry and requires the validator to reject it. A validator that has never rejected anything is untested.

In [ ]:
def validate_registry(rows):
    expected = {(block, allocation) for block in range(8) for allocation in ('breadth', 'balanced', 'phase_depth')} | {(block, 'nearby_jitter') for block in range(4)}
    observed = {(row['block'], row['allocation']) for row in rows}
    if observed != expected or len(rows) != len(expected):
        raise ValueError('registry does not contain the frozen 28 rows')
    for row in rows:
        if row['nominal_catalog_size'] != row['unique_sequences'] * row['origins_per_sequence']:
            raise ValueError('inconsistent nominal catalog')
        if row['nominal_catalog_size'] != 250_000:
            raise ValueError('all rows must have the same nominal catalog')
    for block in range(4):
        phase = next(row for row in rows if row['block'] == block and row['allocation'] == 'phase_depth')
        jitter = next(row for row in rows if row['block'] == block and row['allocation'] == 'nearby_jitter')
        paired_keys = (
            'unique_sequences', 'planned_exposure', 'optimization_seed', 'replicate_seed',
            'sequence_pool_digest', 'source_group_digest', 'phase_catalog_digest',
            'sequence_stream_version', 'phase_stream_version', 'spatial_stream_version',
            'mask_stream_version',
        )
        for key in paired_keys:
            if phase[key] != jitter[key]:
                raise ValueError('phase and jitter must be paired')

validate_registry(registry)
try:
    broken = [dict(row) for row in registry]; broken[-1]['planned_exposure'] = 8_192_000
    validate_registry(broken)
except ValueError:
    pass
else:
    raise AssertionError('mismatched jitter exposure must fail')
try:
    broken = [dict(row) for row in registry]; broken[-1]['sequence_pool_digest'] = digest({'bad': 'pool'})
    validate_registry(broken)
except ValueError:
    pass
else:
    raise AssertionError('mismatched jitter pool must fail')


## Matched continuous gallery margin

The primary contrast subtracts an independent-completion score from a GFC score, so the two must be measured on the same scale or the difference means nothing. The function below is that shared scale: the distance to the nearest competitor minus the distance to the true target, positive when the target wins.

A margin also records how close the call was, which a rank throws away. In the example the two score sets differ in every distance yet produce the same margin, and the assertion makes that explicit.

In [ ]:
def target_margin(distances, target_index):
    distances = np.asarray(distances, dtype=float)
    target = distances[target_index]
    competitor = np.min(np.delete(distances, target_index))
    return float(competitor - target)

gfc_margin = target_margin([0.7, 0.4, 0.9, 1.1], 1)
completion_margin = target_margin([0.8, 0.5, 0.9, 1.2], 1)
assert gfc_margin > 0 and completion_margin > 0
assert np.isclose(gfc_margin - completion_margin, 0.0)


**Takeaway:** reproducibility takes three things working together. The registry has to name the scientific intervention, not just the job. The validation has to run across rows so that the paired comparisons survive. The evaluator has to put its main score and its control on one defined scale. Digests prove identity underneath all three, but they never prove that the design is sound.

Previous: [15. Exposure and replication](15_exposure_and_replication.ipynb) · Next: [17. Iso-catalog allocation](17_hierarchical_support_and_factorial_inference.ipynb)